In [11]:
from ingest import load_faq_data
documents = load_faq_data()

In [12]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [13]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [14]:
documents = documents_llm

In [15]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [16]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [17]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [18]:
from dotenv import load_dotenv
from google import genai
from google.genai.types import HttpOptions

load_dotenv()

PROJECT_ID = "llm-zoomcamp-500505"
LOCATION = "us-central1"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1"),
)

In [19]:
import json
user_prompt = json.dumps(doc)

In [20]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [21]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [22]:
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=data_gen_instructions,
        response_mime_type="application/json",
        response_schema=Questions,
    ),
)

In [23]:
response.parsed.questions
result = Questions.model_validate_json(response.text)
result.questions

['I just found this course, can I still get a certificate?',
 "What's the cutoff date for project submissions if I want a certificate?",
 'Are there specific conditions for receiving a certificate, especially if I join late?',
 "What's the essential requirement for earning a certificate in this course?",
 'If I miss the official submission period for projects, will I still be able to get a certificate?']

In [24]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [25]:
from evaluation_utils import llm_structured

In [27]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Hey, I just heard about this course. Can I still sign up for it?', 'What do I need to do if I want to get a certificate for completing the course?', 'Is there a deadline for submitting the final project to receive a certificate?', 'If I join the course late, can I still earn a certificate?', 'Are there specific requirements for getting the course completion certificate?']


In [28]:
usage

GenerateContentResponseUsageMetadata(
  candidates_token_count=101,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=101
    ),
  ],
  prompt_token_count=179,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=179
    ),
  ],
  thoughts_token_count=731,
  total_token_count=1011,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
)

In [32]:
from evaluation_utils import calc_price

In [34]:
print(usage)

cache_tokens_details=None cached_content_token_count=None candidates_token_count=101 candidates_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=101
)] prompt_token_count=179 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=179
)] thoughts_token_count=731 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=1011 traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>


In [38]:
cost = calc_price(usage)

print(cost)

AttributeError: 'GenerateContentResponseUsageMetadata' object has no attribute 'input_tokens'

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'How is the capstone homework graded, and what do I need to do to get all the points?',
  'document': '0d200c8c58'},
 {'question': 'What are the point values for answering the questions, adding learning resources, and submitting a FAQ question?',
  'document': '0d200c8c58'},
 {'question': 'How many points can I earn from one homework in this project?',
  'document': '0d200c8c58'},
 {'question': 'Does the homework score depend on correct answers only, or are public learning items and FAQ contributions counted too?',
  'document': '0d200c8c58'},
 {'question': 'What tasks should I complete to maximize my homework score and improve my leaderboard position?',
  'document': '0d200c8c58'}]

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame(records)

,question,document
0,"How is the capstone homework graded, and what ...",0d200c8c58
1,What are the point values for answering the qu...,0d200c8c58
2,How many points can I earn from one homework i...,0d200c8c58
3,Does the homework score depend on correct answ...,0d200c8c58
4,What tasks should I complete to maximize my ho...,0d200c8c58


In [ ]:
from evaluation_utils import llm_structured_retry

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
generate_ground_truth(doc)

([{'question': 'How are the capstone homework points calculated?',
   'document': '0d200c8c58'},
  {'question': 'What do I need to do to get full score on a homework?',
   'document': '0d200c8c58'},
  {'question': 'How many points can I earn from one homework assignment in this project?',
   'document': '0d200c8c58'},
  {'question': 'Is there a point breakdown for answers, learning items, and FAQ questions?',
   'document': '0d200c8c58'},
  {'question': 'What actions count toward the homework leaderboard score?',
   'document': '0d200c8c58'}],
 ResponseUsage(input_tokens=263, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=77, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=340))

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/79 [00:00<?, ?it/s]

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

395

In [ ]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.05718675000000001

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.05718675000000001

In [ ]:
df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [ ]:
len(df_ground_truth)

395